# Residential HVAC Change-Point Model

This notebook demonstrates the reusable workflow in `cpm_data.py` and `cpm_model.py`. It prepares one ecobee or EnergyPlus time-step file, builds daily CPM inputs, fits a 3P/4P/5P change-point model, and plots the result.

Update the configuration cell with a local data path before running. Raw building data is intentionally not included in the repository.

## 1. Imports

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from cpm_data import build_cpm_dataset, prepare_ecobee_data, prepare_eplus_data
from cpm_model import fit_cpm, plot_cpm_fit

## 2. Configuration

In [ ]:
INPUT_PATH = Path("building.gz.parquet")
SOURCE_TYPE = "eplus"  # Use "eplus" or "ecobee".
DAYS = 365
MINIMUM_R_SQUARED = 0.1

if SOURCE_TYPE not in {"eplus", "ecobee"}:
    raise ValueError("SOURCE_TYPE must be 'eplus' or 'ecobee'.")
if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Update INPUT_PATH before running: {INPUT_PATH}")

## 3. Prepare time-step data

The preparation function derives HVAC mode, indoor/outdoor temperature differences, and heating/cooling runtime in hours. It also separates the input into daily frames.

In [ ]:
if SOURCE_TYPE == "eplus":
    time_step_data, day_frames = prepare_eplus_data(INPUT_PATH, days=DAYS)
else:
    time_step_data, day_frames = prepare_ecobee_data(INPUT_PATH, days=DAYS)

print(f"Prepared {len(time_step_data):,} time-step rows.")
time_step_data.head()

## 4. Build daily CPM inputs

Heating and cooling runtimes are compared for each day. Mode `1` represents cooling and mode `2` represents heating.

In [ ]:
daily_cpm = build_cpm_dataset(day_frames, days=DAYS)
print(f"Complete regression days: {daily_cpm[['Avg_T_ext', 'Runtimes', 'Mode']].dropna().shape[0]:,}")
daily_cpm.head()

## 5. Fit the change-point model

In [ ]:
plot_data, results, model = fit_cpm(
    daily_cpm,
    value_column="Runtimes",
    minimum_r_squared=MINIMUM_R_SQUARED,
)
results

## 6. Plot the fitted model

In [ ]:
ax = plot_cpm_fit(plot_data)
plt.tight_layout()
plt.show()

## 7. Optional export

Uncomment the following cell to write the daily inputs, fitted coefficients, and figure to an `outputs` directory.

In [ ]:
# output_dir = Path("outputs")
# output_dir.mkdir(exist_ok=True)
# daily_cpm.to_csv(output_dir / "daily_cpm_data.csv", index=False)
# results.to_csv(output_dir / "cpm_results.csv")
# ax.figure.savefig(output_dir / "cpm_fit.png", dpi=180, bbox_inches="tight")